# Naive RAG + Critic Experiment

Runs the Naive RAG pipeline with a critic-revision loop to refine generated answers.
This experiment tests whether post-generation critique and revision can improve
RAG answer quality without changing the retrieval strategy.

**Pipeline:** Query → Dense Retrieval (Cosine) → RAG Generation → Critic → Revision → Answer  
**Base RAG:** Same as EXP-03 (Naive RAG k=5)  
**Critic Loop:** Same as EXP-10 (Vanilla LLM + Critic)  
**Evaluation:** RAGAS and DeepEval metrics

In [5]:
import sys
sys.path.append("..")

import os
import time
import json
import pandas as pd
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from datasets import load_dataset
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
import config
from ast import literal_eval
from deepeval.evaluate import DisplayConfig, AsyncConfig
from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference, BleuScore, RougeScore
from ragas import SingleTurnSample
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import (
    FaithfulnessMetric, ContextualRecallMetric,
    ContextualPrecisionMetric, AnswerRelevancyMetric, GEval
)
import deepeval
import instructor
from groq import AsyncGroq

from ragas.llms.base import InstructorLLM
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()
pd.set_option('display.html.use_mathjax', False)
os.environ["CONFIDENT_TRACE_VERBOSE"] = "0"
os.environ["DEEPEVAL_VERBOSE_MODE"] = "0"
os.environ["DEEPEVAL_RETRY_MAX_ATTEMPTS"] = "2"

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_8575/3329670614.py:17: DeprecationWarning: Importing NonLLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import NonLLMContextRecall
  from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference, BleuScore, RougeScore
/tmp/ipykernel_8575/3329670614.py:17: DeprecationWarning: Importing NonLLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.

In [25]:
import importlib
importlib.reload(config)

<module 'config' from '/content/config.py'>

## Load Vector Stores

In [6]:
def load_chroma(embeddings, db_name=None, persist_dir=None):
    from langchain_chroma import Chroma

    db_name = db_name or f"{config.DEFAULT_EMBEDDING}_pubmed_chromadb"
    persist_dir = persist_dir or str(config.VECTORSTORE_DIR / db_name)
    print(f"Loading ChromaDB from {persist_dir}")
    return Chroma(
        collection_name=db_name,
        persist_directory=persist_dir,
        embedding_function=embeddings,
    )

In [7]:
def get_cosine_retriever(vector_store, k=None):
    k = k or config.TOP_K
    return vector_store.as_retriever(search_kwargs={"k": k})

## LLM & RAG + Critic Chain

In [8]:
def get_groq_llm(model=None, api_key=None):
    return ChatGroq(
        model=model or config.LLM_MODEL,
        api_key=api_key or config.GROQ_API_KEY,
    )


class GroqKeyRotator:
    """Cycles through config.GROQ_API_KEYS, rebuilding the LLM client on each rotation."""

    def __init__(self, model=None):
        if not config.GROQ_API_KEYS:
            raise ValueError("config.GROQ_API_KEYS is empty \u2014 set GROQ_API_KEY or GROQ_API_KEYS in .env")
        self.api_keys = config.GROQ_API_KEYS
        self.model = model or config.LLM_MODEL
        self.current_idx = 0
        print(f"Initialized GroqKeyRotator with {len(self.api_keys)} API key(s)")

    def get_llm(self):
        return ChatGroq(
            model=self.model,
            api_key=self.api_keys[self.current_idx],
        )

    def rotate(self):
        self.current_idx = (self.current_idx + 1) % len(self.api_keys)
        print(f"Rotated to API key index {self.current_idx}")


def is_rate_limit_error(e):
    msg = str(e).lower()
    return any(kw in msg for kw in [
        "rate_limit", "rate limit", "429", "too many requests",
        "tokens per", "token limit", "exceeded",
    ])

## Prompt Templates

- **RAG Prompt**: Same as Naive RAG (EXP-02/03/04) for initial answer generation from retrieved context
- **Critic Prompt**: Same as Vanilla LLM + Critic (EXP-10) for reviewing the generated answer
- **Revision Prompt**: Same as Vanilla LLM + Critic (EXP-10) for refining based on critique

In [9]:
RAG_PROMPT_TEMPLATE = """Use the following research contexts to answer the question.

Context:
{context}

Question: {question}

Answer based only on the provided context. Be precise and evidence-based.

Answer:"""

RAG_PROMPT = PromptTemplate(
    template=RAG_PROMPT_TEMPLATE,
    input_variables=["context", "question"],
)

CRITIC_PROMPT_TEMPLATE = """You are a biomedical expert reviewer trained in evaluation biomedical Question Answering tasks.
Question: {question}

Answer: {answer}

Review the answer for scientific accuracy, completeness, unsupported claims, ambiguity, or missing evidence.
If no major issues exist, say \"No major issues.\" Return proper critique in 2 to 3 sentences only.

Critique:"""

CRITIC_PROMPT = PromptTemplate(
    input_variables=["question", "answer"],
    template=CRITIC_PROMPT_TEMPLATE,
)

REVISION_PROMPT_TEMPLATE = """Revise the answer using the critique.

Question:
{question}

Original Answer:
{answer}

Critique:
{critique}

Provide an improved, precise, evidence-based final answer.

Final Answer:"""

REVISION_PROMPT = PromptTemplate(
    input_variables=["question", "answer", "critique"],
    template=REVISION_PROMPT_TEMPLATE,
)


def build_naive_rag_chain(llm):
    return RAG_PROMPT | llm


def get_tokens(result):
    """Helper Method which returns a tuple of (prompt_tokens, completion_tokens)."""
    usage = getattr(result, "usage_metadata", None) or result.response_metadata.get("token_usage", {})
    prompt = usage.get("prompt_tokens") or usage.get("input_tokens") or 0
    completion = usage.get("completion_tokens") or usage.get("output_tokens") or 0
    return prompt, completion

## RAG + Critic Pipeline Execution

In [10]:
def _run_slice(retriever, slice_df, api_key, model, delay, key_idx):
    """Process a contiguous slice using RAG generation followed by critic-revision.

    Pipeline per question:
        1. Retrieve top-k chunks via cosine similarity
        2. Generate initial answer using RAG prompt + retrieved context
        3. Critic reviews the initial answer for quality issues
        4. Revision refines the answer based on critique

    Args:
        retriever: LangChain retriever (read-only, safe to call from multiple threads).
        slice_df: Contiguous DataFrame slice assigned to this key.
        api_key: Groq API key string dedicated to this thread.
        model: LLM model name.
        delay: Seconds to sleep between rows.
        key_idx: Key index used only for log prefixes.

    Returns:
        Copy of slice_df with new columns: retrieved_contexts, initial_answer,
        critique, generated_answer, total_time, prompt_tokens, completion_tokens, total_tokens.
    """
    llm = ChatGroq(model=model, api_key=api_key)
    rag_chain = build_naive_rag_chain(llm)
    critic_chain = CRITIC_PROMPT | llm
    revision_chain = REVISION_PROMPT | llm

    result_df = slice_df.copy().reset_index(drop=True)
    retrieved_contexts_list = [None] * len(slice_df)
    initial_answers = [None] * len(slice_df)
    critiques = [None] * len(slice_df)
    generated_answer_list = [None] * len(slice_df)
    total_time_list = [None] * len(slice_df)
    prompt_tokens_list = [None] * len(slice_df)
    completion_tokens_list = [None] * len(slice_df)
    total_tokens_list = [None] * len(slice_df)

    for row_idx, (_, row) in enumerate(slice_df.iterrows()):
        question = row["question"]
        try:
            time_start = time.perf_counter()

            # Step 1: Retrieve
            contexts = retriever.invoke(question)
            context_texts = [doc.page_content for doc in contexts]
            context_str = "\n\n".join(context_texts)

            # Step 2: RAG Generation
            rag_result = rag_chain.invoke({"context": context_str, "question": question})
            initial_answer = rag_result.content

            # Step 3: Critic
            critic_result = critic_chain.invoke({"question": question, "answer": initial_answer})
            critique = critic_result.content

            # Step 4: Revision
            revision_result = revision_chain.invoke({
                "question": question,
                "answer": initial_answer,
                "critique": critique,
            })

            time_end = time.perf_counter()

            # Token accounting
            p1, c1 = get_tokens(rag_result)
            p2, c2 = get_tokens(critic_result)
            p3, c3 = get_tokens(revision_result)
            prompt_tokens = p1 + p2 + p3
            completion_tokens = c1 + c2 + c3

            retrieved_contexts_list[row_idx] = context_texts
            initial_answers[row_idx] = initial_answer
            critiques[row_idx] = critique
            generated_answer_list[row_idx] = revision_result.content
            total_time_list[row_idx] = time_end - time_start
            prompt_tokens_list[row_idx] = prompt_tokens
            completion_tokens_list[row_idx] = completion_tokens
            total_tokens_list[row_idx] = prompt_tokens + completion_tokens

        except Exception as e:
            print(f"[Key {key_idx}] Error on '{question[:50]}...': {e}")

        if row_idx < len(slice_df) - 1:
            time.sleep(delay)

    result_df["retrieved_contexts"] = retrieved_contexts_list
    result_df["initial_answer"] = initial_answers
    result_df["critique"] = critiques
    result_df["generated_answer"] = generated_answer_list
    result_df["total_time"] = total_time_list
    result_df["prompt_tokens"] = prompt_tokens_list
    result_df["completion_tokens"] = completion_tokens_list
    result_df["total_tokens"] = total_tokens_list

    completed = sum(1 for x in generated_answer_list if x is not None)
    print(f"[Key {key_idx}] Done \u2014 {completed}/{len(slice_df)} rows collected")
    return result_df


def run_rag_critic_parallel(retriever, df, key_rotator, rows_per_key=None, delay=None):
    """Assign a contiguous slice of rows to each API key and run all slices in parallel.

    Each key runs RAG retrieval + generation + critic + revision in its own thread.

    Args:
        retriever: LangChain retriever instance.
        df: DataFrame with at least question and golden_answer columns.
        key_rotator: GroqKeyRotator providing API keys and LLM model name.
        rows_per_key: Max rows assigned to each key (default config.PARALLEL_BUCKET_SIZE).
        delay: Seconds between rows within each key's slice (default config.PARALLEL_DELAY_SECONDS).

    Returns:
        A copy of df with new columns: retrieved_contexts, initial_answer, critique,
        generated_answer, total_time, prompt_tokens, completion_tokens, total_tokens.
    """
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or config.PARALLEL_DELAY_SECONDS
    api_keys = key_rotator.api_keys

    if len(df) == 0:
        raise ValueError("DataFrame is empty \u2014 nothing to evaluate.")

    total_capacity = len(api_keys) * rows_per_key
    if len(df) > total_capacity:
        print(
            f"Warning: {len(df)} rows exceed capacity ({len(api_keys)} keys \u00d7 {rows_per_key} rows = "
            f"{total_capacity}). Only the first {total_capacity} rows will be processed."
        )
        df = df.iloc[:total_capacity]

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(df):
            break
        slices.append((key, i, df.iloc[start: start + rows_per_key]))

    print(f"\n{len(df)} rows split across {len(slices)} key(s) ({rows_per_key} rows/key max):")
    for key, i, s in slices:
        print(f"  Key {i}: rows {i * rows_per_key}\u2013{i * rows_per_key + len(s) - 1} ({len(s)} rows)")
    print()

    ordered_results = [None] * len(slices)

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(
                _run_slice,
                retriever, s, key, key_rotator.model, delay, i,
            ): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            _, _, s = slices[idx]
            try:
                ordered_results[idx] = future.result()
            except Exception as e:
                print(f"[Key {idx}] Thread failed: {e}")
                fallback = s.copy().reset_index(drop=True)
                fallback["retrieved_contexts"] = [None] * len(s)
                fallback["initial_answer"] = [None] * len(s)
                fallback["critique"] = [None] * len(s)
                fallback["generated_answer"] = [None] * len(s)
                fallback["total_time"] = [None] * len(s)
                fallback["prompt_tokens"] = [None] * len(s)
                fallback["completion_tokens"] = [None] * len(s)
                fallback["total_tokens"] = [None] * len(s)
                ordered_results[idx] = fallback

    final_df = pd.concat(ordered_results, ignore_index=True)
    completed = final_df["generated_answer"].notna().sum()
    print(f"\nCompleted {completed}/{len(df)} questions total")
    valid_times = final_df["total_time"].dropna()
    valid_tokens = final_df["total_tokens"].dropna()
    if len(valid_times) > 0:
        print(f"Average Time Per Query: {valid_times.mean():.2f}s")
    if len(valid_tokens) > 0:
        print(f"Average Total Tokens Per Query: {valid_tokens.mean():.0f}")
    return final_df

## Evaluation Functions

In [10]:
class GroqModel(DeepEvalBaseLLM):
    def __init__(self, model=None):
        self.model = model or get_groq_llm()

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        response = self.model.invoke(prompt)
        return response.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return "Groq Model"


def build_test_cases(eval_df):
    return [
        LLMTestCase(
            input=row["question"],
            actual_output=row["generated_answer"],
            retrieval_context=row["retrieved_contexts"],
            expected_output=row["golden_answer"],
        )
        for _, row in eval_df.iterrows()
    ]


def _make_ragas_scores_df(all_scores, metric_name):
    return pd.DataFrame({
        "question_index": range(len(all_scores)),
        metric_name: all_scores,
    })


def evaluate_ragas(eval_df, metric, results_file=None):
    metric_name = type(metric).__name__
    all_scores = []

    for _, row in eval_df.iterrows():
        sample = SingleTurnSample(
            user_input=row["question"],
            retrieved_contexts=list(row["retrieved_contexts"]),
            reference_contexts=row["golden_contexts"],
            reference=row["golden_answer"],
            response=row["generated_answer"]
        )
        score = metric.single_turn_score(sample)
        all_scores.append(score)

    avg = sum(all_scores) / len(all_scores)
    print(f"\n=== {metric_name}: {avg:.4f} (avg over {len(all_scores)} samples) ===")

    scores_df = _make_ragas_scores_df(all_scores, metric_name)

    if results_file:
        scores_df.to_csv(results_file, index=False)
        print(f"Saved scores to {results_file}")

    return all_scores, avg, scores_df


def build_ragas_combined(eval_df, score_dfs, results_file=None):
    combined = eval_df.copy().reset_index(drop=True)
    combined.insert(0, "question_index", range(len(combined)))
    combined = combined.rename(columns={"generated_answer": "generated_response"})

    for scores_df in score_dfs:
        combined = combined.merge(scores_df, on="question_index", how="left")

    if results_file:
        combined.to_csv(results_file, index=False)
        print(f"Saved combined RAGAS results to {results_file}")

    return combined


def _run_deepeval_slice(test_case_slice, api_key, model, metric_cls, threshold, delay, key_idx, metric_kwargs=None):
    llm = ChatGroq(model=model, api_key=api_key)
    results = []

    for i, test_case in enumerate(test_case_slice):
        try:
            metric = metric_cls(threshold=threshold, model=GroqModel(model=llm),
                                **metric_kwargs)
            result = deepeval.evaluate([test_case], metrics=[metric],
                                       display_config= DisplayConfig(
                                           verbose_mode=False,
                                           show_indicator=False,
                                           print_results=False),
                                       async_config=AsyncConfig(run_async=False))
            results.extend(result.test_results)
        except Exception as e:
            print(f"[Key {key_idx}] Error on case {i + 1}/{len(test_case_slice)}: {e}")

        if i < len(test_case_slice) - 1:
            time.sleep(delay)

    print(f"[Key {key_idx}] Done \u2014 {len(results)}/{len(test_case_slice)} cases evaluated")
    return results


def evaluate_deepeval_parallel(test_cases, metric_cls, key_rotator, threshold=0.5, results_file=None, delay=None, rows_per_key=None, metric_kwargs=None):
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or config.DEEPEVAL_DELAY_SECONDS
    api_keys = key_rotator.api_keys
    metric_kwargs = {} if metric_kwargs is None else dict(metric_kwargs)

    if not test_cases:
        raise ValueError("test_cases is empty \u2014 nothing to evaluate.")

    total_capacity = len(api_keys) * rows_per_key
    if len(test_cases) > total_capacity:
        print(
            f"Warning: {len(test_cases)} cases exceed capacity ({len(api_keys)} keys \u00d7 {rows_per_key} = "
            f"{total_capacity}). Only the first {total_capacity} cases will be processed."
        )
        test_cases = test_cases[:total_capacity]

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(test_cases):
            break
        slices.append((key, i, test_cases[start: start + rows_per_key]))

    print(f"\n{len(test_cases)} cases split across {len(slices)} key(s) ({rows_per_key} cases/key max):")
    for key, i, s in slices:
        print(f"  Key {i}: cases {i * rows_per_key}\u2013{i * rows_per_key + len(s) - 1} ({len(s)} cases)")
    print()

    ordered_results = [None] * len(slices)

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(
                _run_deepeval_slice,
                s, key, key_rotator.model,
                metric_cls, threshold, delay, i, metric_kwargs
            ): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            try:
                ordered_results[idx] = future.result()
            except Exception as e:
                print(f"[Key {idx}] Thread failed: {e}")
                ordered_results[idx] = []

    all_results = []
    res_df = None
    for key_results in ordered_results:
        all_results.extend(key_results)

    if all_results:
        scores = [r.metrics_data[0].score for r in all_results]
        metric_name = all_results[0].metrics_data[0].name
        average = sum(scores) / len(scores)
        print(f"\n=== {metric_name}: {average:.4f} (avg over {len(scores)} samples) ===")

        rows = []
        for r in all_results:
            rows.append({
                "question": r.input,
                "generated_answer": r.actual_output,
                "retrieved_contexts": r.retrieval_context,
                "golden_answer": r.expected_output,
                r.metrics_data[0].name: r.metrics_data[0].score,
            })
        res_df = pd.DataFrame(rows)
        if results_file:
            res_df.to_csv(results_file, index=False)
            print(f"Saved to {results_file}")

    return all_results, res_df


def resume_deepeval_from_csv(eval_dataset, existing_results_file, metric_cls, key_rotator,
    metric_column, threshold=0.5, delay=None, rows_per_key=None, metric_kwargs=None):
    existing_df = pd.read_csv(existing_results_file)

    completed_questions = set(existing_df.loc[
        existing_df[metric_column].notna(),"question"].astype(str))

    missing_df = eval_dataset[~eval_dataset["question"].astype(str)
                              .isin(completed_questions)].copy()
    if missing_df.empty:
        print("All rows already completed.")
        return existing_df
    print(f"Need to recompute {len(missing_df)} rows.")

    test_cases = build_test_cases(missing_df)

    _, new_results_df = evaluate_deepeval_parallel(test_cases, metric_cls,
        key_rotator, threshold=threshold, delay=delay, rows_per_key=rows_per_key,
        metric_kwargs=metric_kwargs)

    existing_df = existing_df[~existing_df["question"].astype(str).isin(
            new_results_df["question"].astype(str))]

    final_df = pd.concat([existing_df, new_results_df], ignore_index=True)
    if "question_idx" in final_df.columns:
        final_df = final_df.drop(columns=["question_idx"])
    final_df = final_df.merge(eval_dataset[["question", "question_idx"]], on="question", how="left")
    cols = ["question_idx"] + [c for c in final_df.columns if c != "question_idx"]
    final_df = final_df.sort_values("question_idx").reset_index(drop=True)

    final_df.to_csv(existing_results_file, index=False)
    print(f"Completed: {len(final_df)}/{len(eval_dataset)} rows")

    return final_df

---
## Setup

In [12]:
embedding_key = config.DEFAULT_EMBEDDING
embeddings = HuggingFaceEmbeddings(model_name=config.EMBEDDING_MODELS[embedding_key])
key_rotator = GroqKeyRotator()
llm = key_rotator.get_llm()
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Embedding: {config.EMBEDDING_MODELS[embedding_key]}")
print(f"LLM: {key_rotator.model}")
print(f"Run timestamp: {timestamp}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initialized GroqKeyRotator with 10 API key(s)
Embedding: sentence-transformers/all-MiniLM-L6-v2
LLM: llama-3.3-70b-versatile
Run timestamp: 20260517_204517


## Prepare Evaluation Sample

Reads the pre-built golden dataset from `data/processed/golden_dataset_complete.csv`.
Generate this file once by running `python3 sampling.py` from the `research/` directory.
Keeping the split fixed is critical — regenerating mid-experiment would change which
questions each RAG variant sees, invalidating cross-experiment comparisons.

In [13]:
golden_df = pd.read_csv(config.DATA_PROCESSED_DIR / "golden_dataset_complete.csv")
print(f"Loaded {len(golden_df)} samples from golden_dataset_complete.csv")
golden_df['golden_contexts'] = golden_df['golden_contexts'].apply(literal_eval)
golden_df.info()

Loaded 200 samples from golden_dataset_complete.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   question_idx     200 non-null    int64 
 1   question         200 non-null    object
 2   golden_answer    200 non-null    object
 3   golden_contexts  200 non-null    object
 4   query_type       200 non-null    object
 5   pubids_needed    200 non-null    object
dtypes: int64(1), object(5)
memory usage: 9.5+ KB


## Load Vector Store

In [14]:
vector_store = load_chroma(embeddings, db_name=f"{embedding_key}_pubmed_chromadb")

Loading ChromaDB from /content/vectorstores/minilm_pubmed_chromadb


## Run Naive RAG + Critic (k=5)

In [15]:
cosine_retriever = get_cosine_retriever(vector_store, k=5)

### Smoke Test (1 row)

In [16]:
eval_test = run_rag_critic_parallel(cosine_retriever, golden_df.head(1), key_rotator)
eval_test[["question", "initial_answer", "critique", "generated_answer", "total_time", "total_tokens"]]


1 rows split across 1 key(s) (20 rows/key max):
  Key 0: rows 0–0 (1 rows)

[Key 0] Done — 1/1 rows collected

Completed 1/1 questions total
Average Time Per Query: 2.84s
Average Total Tokens Per Query: 2101


,question,initial_answer,critique,generated_answer,total_time,total_tokens
0,Is there a relationship between rheumatoid art...,"Yes, there is a relationship between rheumatoi...","The answer is generally accurate, but it lacks...","Yes, there is a significant relationship betwe...",2.843054,2101


### Full Run (200 questions)

In [17]:
eval_dataset = run_rag_critic_parallel(cosine_retriever, golden_df, key_rotator)
eval_dataset.to_csv(str(config.RESULTS_EVALSETS_DIR / f"naive_rag_critic_{embedding_key}_chroma_cosine_{timestamp}.csv"))
print(f"Generated {len(eval_dataset)} answers")


200 rows split across 10 key(s) (20 rows/key max):
  Key 0: rows 0–19 (20 rows)
  Key 1: rows 20–39 (20 rows)
  Key 2: rows 40–59 (20 rows)
  Key 3: rows 60–79 (20 rows)
  Key 4: rows 80–99 (20 rows)
  Key 5: rows 100–119 (20 rows)
  Key 6: rows 120–139 (20 rows)
  Key 7: rows 140–159 (20 rows)
  Key 8: rows 160–179 (20 rows)
  Key 9: rows 180–199 (20 rows)

[Key 4] Done — 20/20 rows collected
[Key 2] Done — 20/20 rows collected
[Key 1] Done — 20/20 rows collected
[Key 3] Done — 20/20 rows collected
[Key 6] Done — 20/20 rows collected
[Key 0] Done — 20/20 rows collected
[Key 5] Done — 20/20 rows collected
[Key 9] Done — 20/20 rows collected
[Key 7] Done — 20/20 rows collected
[Key 8] Done — 20/20 rows collected

Completed 200/200 questions total
Average Time Per Query: 2.86s
Average Total Tokens Per Query: 1900
Generated 200 answers


In [ ]:
error_rows = eval_dataset[eval_dataset['generated_answer'].isnull()]
print(f"Errors: {len(error_rows)}")
error_rows

In [ ]:
# Re-run for error rows if any
if len(error_rows) > 0:
    sub_golden_df = golden_df[golden_df['question_idx'].isin(error_rows['question_idx'])]
    sub_golden_df = sub_golden_df.reset_index(drop=True)
    sub_eval_dataset = run_rag_critic_parallel(cosine_retriever, sub_golden_df, key_rotator, rows_per_key=3)

    for idx, row in sub_eval_dataset.iterrows():
        eval_dataset.at[row['question_idx'], 'retrieved_contexts'] = row['retrieved_contexts']
        eval_dataset.at[row['question_idx'], 'initial_answer'] = row['initial_answer']
        eval_dataset.at[row['question_idx'], 'critique'] = row['critique']
        eval_dataset.at[row['question_idx'], 'generated_answer'] = row['generated_answer']
        eval_dataset.at[row['question_idx'], 'total_time'] = row['total_time']
        eval_dataset.at[row['question_idx'], 'prompt_tokens'] = row['prompt_tokens']
        eval_dataset.at[row['question_idx'], 'completion_tokens'] = row['completion_tokens']
        eval_dataset.at[row['question_idx'], 'total_tokens'] = row['total_tokens']

    eval_dataset.to_csv(str(config.RESULTS_EVALSETS_DIR / f"naive_rag_critic_{embedding_key}_chroma_cosine_{timestamp}.csv"))
    print(f"Remaining errors: {eval_dataset['generated_answer'].isnull().sum()}")
else:
    print("No errors to re-run.")

---
## RAGAS Evaluation

In [18]:
ragas_cr_scores, ragas_cr_avg, ragas_cr_df = evaluate_ragas(
    eval_dataset, NonLLMContextRecall(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"naive_rag_critic_{embedding_key}_context_recall_{timestamp}.csv")
)


=== NonLLMContextRecall: 0.2008 (avg over 200 samples) ===
Saved scores to /content/results/ragas/naive_rag_critic_minilm_context_recall_20260517_204517.csv


In [19]:
ragas_cp_scores, ragas_cp_avg, ragas_cp_df = evaluate_ragas(
    eval_dataset, NonLLMContextPrecisionWithReference(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"naive_rag_critic_{embedding_key}_context_precision_{timestamp}.csv")
)


=== NonLLMContextPrecisionWithReference: 0.3067 (avg over 200 samples) ===
Saved scores to /content/results/ragas/naive_rag_critic_minilm_context_precision_20260517_204517.csv


In [20]:
ragas_bleu_scores, ragas_bleu_avg, ragas_bleu_df = evaluate_ragas(
    eval_dataset, BleuScore(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"naive_rag_critic_{embedding_key}_bleu_{timestamp}.csv")
)


=== BleuScore: 0.1344 (avg over 200 samples) ===
Saved scores to /content/results/ragas/naive_rag_critic_minilm_bleu_20260517_204517.csv


In [21]:
ragas_rouge_scores, ragas_rouge_avg, ragas_rouge_df = evaluate_ragas(
    eval_dataset, RougeScore(rouge_type="rougeL", mode="fmeasure"),
    results_file=str(config.RESULTS_RAGAS_DIR / f"naive_rag_critic_{embedding_key}_rouge_{timestamp}.csv")
)


=== RougeScore: 0.1510 (avg over 200 samples) ===
Saved scores to /content/results/ragas/naive_rag_critic_minilm_rouge_20260517_204517.csv


In [22]:
combined_ragas = build_ragas_combined(
    eval_dataset, [ragas_cr_df, ragas_cp_df, ragas_bleu_df, ragas_rouge_df],
    results_file=str(config.RESULTS_RAGAS_DIR / f"naive_rag_critic_{embedding_key}_combined_{timestamp}.csv")
)

Saved combined RAGAS results to /content/results/ragas/naive_rag_critic_minilm_combined_20260517_204517.csv


---
## DeepEval Evaluation

In [12]:
timestamp = "20260517_204517"
embedding_key = config.DEFAULT_EMBEDDING
eval_dataset = pd.read_csv(str(config.RESULTS_EVALSETS_DIR / f"naive_rag_critic_{embedding_key}_chroma_cosine_{timestamp}.csv"))
eval_dataset['golden_contexts'] = eval_dataset['golden_contexts'].apply(literal_eval)
eval_dataset['retrieved_contexts'] = eval_dataset['retrieved_contexts'].apply(literal_eval)

eval_dataset.head(2)

,Unnamed: 0,question_idx,question,golden_answer,golden_contexts,query_type,pubids_needed,retrieved_contexts,initial_answer,critique,generated_answer,total_time,prompt_tokens,completion_tokens,total_tokens
0,0,0,Is there a relationship between rheumatoid art...,Based on data derived from self-reported healt...,"[1,412 individuals attending the University of...",Single-hop,['10783841'],[CONCLUSIONS: Based on data derived from self-...,"Yes, there is a relationship between rheumatoi...","The answer is generally accurate, but it lacks...","Yes, there is a significant relationship betwe...",3.437045,1304,1017,2321
1,1,1,"Do the changes in the serum levels of IL-2, IL...",The enhancement of serum TNFalpha and IL-6 lev...,[Acute pancreatitis is the major complication ...,Single-hop,['18670651'],[RESULTS: Seven of the 45 patients (15.5%) dev...,"According to the provided context, the enhance...",The answer is mostly accurate but lacks specif...,The changes in serum levels of certain cytokin...,2.598130,1416,753,2169


In [13]:
test_cases = build_test_cases(eval_dataset)

de_key_rotator = GroqKeyRotator(model="openai/gpt-oss-120b")

Initialized GroqKeyRotator with 10 API key(s)


In [ ]:
deepeval_cr, de_cr_df = evaluate_deepeval_parallel(
    test_cases, ContextualRecallMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"naive_rag_critic_{embedding_key}_ctx_recall_{timestamp}.csv")
)

In [ ]:
deepeval_cp, de_cp_df = evaluate_deepeval_parallel(
    test_cases, ContextualPrecisionMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"naive_rag_critic_{embedding_key}_ctx_precision_{timestamp}.csv")
)

In [27]:
deepeval_f, de_f_df = evaluate_deepeval_parallel(
    test_cases, FaithfulnessMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"naive_rag_critic_{embedding_key}_faithfulness_{timestamp}.csv"),
)


200 cases split across 10 key(s) (20 cases/key max):
  Key 0: cases 0–19 (20 cases)
  Key 1: cases 20–39 (20 cases)
  Key 2: cases 40–59 (20 cases)
  Key 3: cases 60–79 (20 cases)
  Key 4: cases 80–99 (20 cases)
  Key 5: cases 100–119 (20 cases)
  Key 6: cases 120–139 (20 cases)
  Key 7: cases 140–159 (20 cases)
  Key 8: cases 160–179 (20 cases)
  Key 9: cases 180–199 (20 cases)



[Key 7] Done — 20/20 cases evaluated

=== Faithfulness: 0.9448 (avg over 197 samples) ===
Saved to /content/results/deepeval/naive_rag_critic_minilm_faithfulness_20260517_204517.csv


In [28]:
faithfulness_df = resume_deepeval_from_csv(
    eval_dataset, str(config.RESULTS_DEEPEVAL_DIR / f"naive_rag_critic_{embedding_key}_faithfulness_{timestamp}.csv"),
    FaithfulnessMetric, de_key_rotator, "Faithfulness", rows_per_key=1
)

Need to recompute 3 rows.

3 cases split across 3 key(s) (1 cases/key max):
  Key 0: cases 0–0 (1 cases)
  Key 1: cases 1–1 (1 cases)
  Key 2: cases 2–2 (1 cases)



[Key 2] Done — 1/1 cases evaluated

=== Faithfulness: 0.9762 (avg over 3 samples) ===
Completed: 200/200 rows


In [29]:
faithfulness_df['Faithfulness'].apply('mean')

np.float64(0.9452597035317624)

In [31]:
evaluation_steps = [
    "Compare the generated answer with the reference answer in the context of the original biomedical question.",
    "Check whether the generated answer contains factually correct biomedical information and no contradictions to the reference answer.",
    "Verify that all clinically important facts needed to answer the question are present and no critical information is missing.",
    "Ignore wording differences, but penalize incorrect medical claims, unsupported conclusions, or misleading clinical interpretations."
]
deepeval_ac, de_ac_df = evaluate_deepeval_parallel(
    test_cases, GEval, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"naive_rag_critic_{embedding_key}_answer_correctness_{timestamp}.csv"),
    metric_kwargs={
        "name": "AnswerCorrectness",
        "evaluation_steps": evaluation_steps,
        "evaluation_params": [
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
            LLMTestCaseParams.EXPECTED_OUTPUT,
        ]}
)


200 cases split across 10 key(s) (20 cases/key max):
  Key 0: cases 0–19 (20 cases)
  Key 1: cases 20–39 (20 cases)
  Key 2: cases 40–59 (20 cases)
  Key 3: cases 60–79 (20 cases)
  Key 4: cases 80–99 (20 cases)
  Key 5: cases 100–119 (20 cases)
  Key 6: cases 120–139 (20 cases)
  Key 7: cases 140–159 (20 cases)
  Key 8: cases 160–179 (20 cases)
  Key 9: cases 180–199 (20 cases)



[Key 8] Done — 20/20 cases evaluated

=== AnswerCorrectness [GEval]: 0.6583 (avg over 199 samples) ===
Saved to /content/results/deepeval/naive_rag_critic_minilm_answer_correctness_20260517_204517.csv


In [32]:
answer_correctness_df = resume_deepeval_from_csv(
    eval_dataset, str(config.RESULTS_DEEPEVAL_DIR / f"naive_rag_critic_{embedding_key}_answer_correctness_{timestamp}.csv"),
    GEval, de_key_rotator, "AnswerCorrectness [GEval]", rows_per_key=1,
    metric_kwargs={
        "name": "AnswerCorrectness",
        "evaluation_steps": evaluation_steps,
        "evaluation_params": [
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
            LLMTestCaseParams.EXPECTED_OUTPUT,
        ]}
)

Need to recompute 1 rows.

1 cases split across 1 key(s) (1 cases/key max):
  Key 0: cases 0–0 (1 cases)



[Key 0] Done — 1/1 cases evaluated

=== AnswerCorrectness [GEval]: 1.0000 (avg over 1 samples) ===
Completed: 200/200 rows


In [33]:
answer_correctness_df['AnswerCorrectness [GEval]'].apply('mean')

np.float64(0.66)

In [14]:
deepeval_ar, de_ar_df = evaluate_deepeval_parallel(
    test_cases, AnswerRelevancyMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"naive_rag_critic_{embedding_key}_ans_relevancy_{timestamp}.csv")
)


200 cases split across 10 key(s) (20 cases/key max):
  Key 0: cases 0–19 (20 cases)
  Key 1: cases 20–39 (20 cases)
  Key 2: cases 40–59 (20 cases)
  Key 3: cases 60–79 (20 cases)
  Key 4: cases 80–99 (20 cases)
  Key 5: cases 100–119 (20 cases)
  Key 6: cases 120–139 (20 cases)
  Key 7: cases 140–159 (20 cases)
  Key 8: cases 160–179 (20 cases)
  Key 9: cases 180–199 (20 cases)



[Key 5] Done — 20/20 cases evaluated

=== Answer Relevancy: 0.9592 (avg over 200 samples) ===
Saved to /content/results/deepeval/naive_rag_critic_minilm_ans_relevancy_20260517_204517.csv
